<h1 style="color: #FECB05; text-align: center;">Fine-Tuning y Transfer Learning</h1>


<h2 style="color: #007ACC;">Autores</h2>


- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)


---


<h2 style="color: #007ACC;">Tabla de contenido</h2>


- [<span style="color: #005C99;">Introducción</span>](#introduction)
- [<span style="color: #005C99;">Transfer Learning</span>](#transfer-learning)
- [<span style="color: #005C99;">Fine-Tuning</span>](#fine-tuning)
- [<span style="color: #005C99;">Ejemplo en Keras: gatos vs perros (CIFAR-10)</span>](#ejemplo)
- [<span style="color: #005C99;">Fase 1: Transfer Learning (base congelada)</span>](#fase-tl)
- [<span style="color: #005C99;">Fase 2: Fine-Tuning parcial</span>](#fase-ft)
- [<span style="color: #005C99;">Comparación y diagnóstico</span>](#comparacion)
- [<span style="color: #005C99;">Cuándo usar cada enfoque</span>](#eleccion)
- [<span style="color: #005C99;">Problemas comunes</span>](#problemas)
- [<span style="color: #005C99;">Referencias</span>](#reference)


---


<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.


In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib scikit-learn tensorflow


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input, RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)


In [ ]:
def plot_history(history, title="Curvas de entrenamiento"):
    hist = history.history
    epochs = range(1, len(hist["loss"]) + 1)

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, hist["loss"], label="train loss")
    if "val_loss" in hist:
        plt.plot(epochs, hist["val_loss"], label="val loss")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title(title + " — loss")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    acc_key = "accuracy" if "accuracy" in hist else "binary_accuracy"
    val_key = "val_" + acc_key
    if acc_key in hist:
        plt.plot(epochs, hist[acc_key], label="train acc")
        if val_key in hist:
            plt.plot(epochs, hist[val_key], label="val acc")
        plt.xlabel("Época")
        plt.ylabel("Accuracy")
        plt.title(title + " — accuracy")
        plt.legend()
        plt.grid(True)

    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(cm, classes, title="Matriz de confusión"):
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(classes))
    plt.xticks(ticks, classes, rotation=45)
    plt.yticks(ticks, classes)

    thresh = cm.max() / 2.0 if cm.size else 0
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], "d"),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel("Etiqueta real")
    plt.xlabel("Predicción")
    plt.tight_layout()
    plt.show()


def evaluate_binary(model, ds, class_names=("gato", "perro")):
    y_true, y_prob = [], []
    for xb, yb in ds:
        y_true.append(yb.numpy().ravel())
        y_prob.append(model.predict(xb, verbose=0).ravel())
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)

    print(classification_report(y_true, y_pred, target_names=list(class_names), digits=3))
    cm = confusion_matrix(y_true, y_pred)
    plot_confusion_matrix(cm, class_names)
    return y_true, y_pred, y_prob


<a id="introduction"></a>
<h2 style="color: #007ACC;">Introducción</h2>


Entrenar una red profunda **desde cero** suele exigir muchos datos etiquetados, tiempo y cómputo. En la práctica, casi siempre partimos de un modelo **preentrenado** (por ejemplo en ImageNet) y lo adaptamos a nuestra tarea.

Dos ideas centrales:

1. **Transfer Learning / feature extraction:** reutilizamos la base convolucional como extractor de características y entrenamos solo una cabeza nueva.
2. **Fine-Tuning:** además, descongelamos (parte de) la base y la ajustamos con una tasa de aprendizaje **baja**.

El objetivo de este cuaderno es dejar clara la diferencia conceptual y practicar el flujo recomendado en Keras: primero entrenar la cabeza con la base congelada, y después hacer fine-tuning parcial.


<a id="transfer-learning"></a>
<h2 style="color: #007ACC;">Transfer Learning</h2>


El *transfer learning* aprovecha que las capas tempranas de una CNN aprenden patrones genéricos (bordes, texturas, formas simples) útiles en muchas tareas de visión. En lugar de aprender esos filtros otra vez, los **transferimos** desde un modelo preentrenado.

En el sentido más estricto de *feature extraction*:

- Cargamos una arquitectura con pesos de ImageNet (`include_top=False`).
- **Congelamos** toda la base (`trainable = False`).
- Añadimos una cabeza propia (pooling + capas densas).
- Entrenamos **solo** la cabeza.

Ventajas típicas: menos datos, menos épocas, menor riesgo de destruir representaciones útiles.

<img src="../img/tl_ft.svg" alt="Transfer Learning vs Fine-Tuning" width="860">

> En la literatura a veces se usa “transfer learning” como término paraguas (incluye fine-tuning). Aquí separamos **extracción de características** y **fine-tuning** para que el código refleje dos fases distintas.


<a id="fine-tuning"></a>
<h2 style="color: #007ACC;">Fine-Tuning</h2>


El **fine-tuning** ajusta (parte de) los pesos preentrenados a la tarea objetivo. La intuición: las capas altas capturan conceptos más específicos del dominio de origen; conviene adaptarlas cuando el dominio objetivo difiere o cuando ya tenemos una cabeza estable.

### Fine-tuning total
Se descongelan **todas** las capas y se entrena el modelo completo con LR baja. Útil con muchos datos y dominio lejano; costoso y con más riesgo de sobreajuste.

### Fine-tuning parcial (el más usado)
Se mantienen congeladas las capas bajas y se descongelan solo las superiores (por bloque). Es el equilibrio habitual entre adaptación y estabilidad.

### Receta práctica (2 fases)
1. Entrenar la cabeza con la base congelada.
2. Descongelar las últimas capas / último bloque.
3. Recompilar con **LR más baja** (p. ej. 10× menor).
4. Continuar el entrenamiento con early stopping.


<a id="ejemplo"></a>
<h2 style="color: #007ACC;">Ejemplo en Keras: gatos vs perros (CIFAR-10)</h2>


Usaremos un subconjunto binario de **CIFAR-10** (clases *cat* y *dog*). Ventajas didácticas:

- Se descarga con Keras (no depende de carpetas locales del repositorio).
- Es lo bastante pequeño para experimentar en CPU/Colab.
- ImageNet ya vio gatos y perros, así que el *transfer* es natural; el fine-tuning suele afinar detalles de resolución/estilo.

Trabajaremos con **MobileNetV2** (rápida y con `preprocess_input` propio). La misma receta aplica a VGG16, ResNet, EfficientNet, etc.


In [ ]:
# Hiperparámetros
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS_TL = 12
EPOCHS_FT = 12
LR_TL = 1e-3
LR_FT = 1e-4
CLASS_IDS = (3, 5)          # cat, dog en CIFAR-10
CLASS_NAMES = ("gato", "perro")
MAX_TRAIN = 4000            # submuestra para acelerar la demo
MAX_TEST = 1000


In [ ]:
def load_cats_dogs(max_train=MAX_TRAIN, max_test=MAX_TEST):
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
    y_train = y_train.ravel()
    y_test = y_test.ravel()

    def filter_binary(x, y):
        mask = np.isin(y, CLASS_IDS)
        x, y = x[mask], y[mask]
        # Remapeo: cat -> 0, dog -> 1
        y = (y == CLASS_IDS[1]).astype(np.float32)
        return x, y

    x_train, y_train = filter_binary(x_train, y_train)
    x_test, y_test = filter_binary(x_test, y_test)

    # Submuestreo estratificado
    if len(x_train) > max_train:
        x_train, _, y_train, _ = train_test_split(
            x_train, y_train, train_size=max_train, stratify=y_train, random_state=SEED
        )
    if len(x_test) > max_test:
        x_test, _, y_test, _ = train_test_split(
            x_test, y_test, train_size=max_test, stratify=y_test, random_state=SEED
        )

    # Split train / val
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train, test_size=0.2, stratify=y_train, random_state=SEED
    )
    return (x_tr, y_tr), (x_val, y_val), (x_test, y_test)


(x_tr, y_tr), (x_val, y_val), (x_te, y_te) = load_cats_dogs()
print("Train:", x_tr.shape, "| Val:", x_val.shape, "| Test:", x_te.shape)
print("Proporción perro (train):", float(y_tr.mean()))


In [ ]:
def make_dataset(x, y, training=False):
    """Redimensiona a IMG_SIZE y deja píxeles en [0, 255] (float).
    El preprocess_input de MobileNetV2 se aplica dentro del modelo,
    después de la aumentación.
    """
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(buffer_size=len(x), seed=SEED, reshuffle_each_iteration=True)

    def _prep(image, label):
        image = tf.image.resize(tf.cast(image, tf.float32), (IMG_SIZE, IMG_SIZE))
        return image, label

    ds = ds.map(_prep, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(x_tr, y_tr, training=True)
val_ds = make_dataset(x_val, y_val, training=False)
test_ds = make_dataset(x_te, y_te, training=False)

# Vista rápida
fig, axes = plt.subplots(2, 6, figsize=(10, 3.5))
for ax, img, lab in zip(axes.ravel(), x_tr[:12], y_tr[:12]):
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[int(lab)], fontsize=9)
    ax.axis("off")
plt.suptitle("Muestras CIFAR-10: gato vs perro")
plt.tight_layout()
plt.show()


### Augmentation y preprocesado

Orden recomendado:

1. Redimensionar (en el `tf.data.Dataset`).
2. Aumentar en entrenamiento (capas `Random*` del modelo; en inferencia se desactivan).
3. Aplicar `preprocess_input` de MobileNetV2 (escala típica a ~[-1, 1]).

Así la aumentación geométrica actúa sobre la imagen “natural” y el backbone recibe exactamente el preprocesado con el que fue preentrenado.


In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        RandomFlip("horizontal"),
        RandomRotation(0.05),
        RandomZoom(0.1),
    ],
    name="data_augmentation",
)


def build_model(base_trainable=False):
    base = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base.trainable = base_trainable

    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = preprocess_input(x)
    # training=False: BN de la base en modo inferencia (estable con TL / inicio de FT)
    x = base(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation="sigmoid")(x)

    model = Model(inputs, outputs, name="mobilenet_cats_dogs")
    return model, base


model, base_model = build_model(base_trainable=False)
model.compile(
    optimizer=Adam(learning_rate=LR_TL),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()


<a id="fase-tl"></a>
<h2 style="color: #007ACC;">Fase 1: Transfer Learning (base congelada)</h2>


En esta fase la base de MobileNetV2 permanece congelada: solo aprende la cabeza. Es el paso correcto **antes** de descongelar capas.


In [ ]:
callbacks_tl = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6),
]

history_tl = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_TL,
    callbacks=callbacks_tl,
)
plot_history(history_tl, title="Fase 1 — Transfer Learning")


In [ ]:
print("=== Evaluación en test — Transfer Learning ===")
_ = evaluate_binary(model, test_ds, CLASS_NAMES)
tl_test = model.evaluate(test_ds, verbose=0)
print(f"Test loss={tl_test[0]:.4f} | Test acc={tl_test[1]:.4f}")


<a id="fase-ft"></a>
<h2 style="color: #007ACC;">Fase 2: Fine-Tuning parcial</h2>


Descongelamos las **últimas capas** de la base (no todo el modelo) y bajamos la tasa de aprendizaje. Evitamos descongelar capas de BatchNormalization de forma agresiva: al llamar la base con `training=False` en la construcción, las estadísticas de BN se mantienen estables; al fine-tunear, recompilamos y permitimos actualizar pesos de convolución del tope.


In [ ]:
# Descongelar la base y dejar entrenables solo las últimas capas
base_model.trainable = True

# Congelar todo excepto el tramo final
FINE_TUNE_AT = len(base_model.layers) - 40  # ~último bloque / capas altas
for i, layer in enumerate(base_model.layers):
    layer.trainable = i >= FINE_TUNE_AT

trainable_count = sum(1 for layer in base_model.layers if layer.trainable)
frozen_count = sum(1 for layer in base_model.layers if not layer.trainable)
print(f"Capas base entrenables: {trainable_count} | congeladas: {frozen_count}")
print(f"Umbral FINE_TUNE_AT = {FINE_TUNE_AT} / {len(base_model.layers)}")

# Recompilar con LR más baja (imprescindible)
model.compile(
    optimizer=Adam(learning_rate=LR_FT),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

callbacks_ft = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7),
]

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    callbacks=callbacks_ft,
)
plot_history(history_ft, title="Fase 2 — Fine-Tuning")


In [ ]:
print("=== Evaluación en test — Fine-Tuning ===")
_ = evaluate_binary(model, test_ds, CLASS_NAMES)
ft_test = model.evaluate(test_ds, verbose=0)
print(f"Test loss={ft_test[0]:.4f} | Test acc={ft_test[1]:.4f}")


<a id="comparacion"></a>
<h2 style="color: #007ACC;">Comparación y diagnóstico</h2>


In [ ]:
summary = pd.DataFrame(
    [
        {"fase": "1. Transfer Learning", "test_loss": tl_test[0], "test_accuracy": tl_test[1]},
        {"fase": "2. Fine-Tuning", "test_loss": ft_test[0], "test_accuracy": ft_test[1]},
    ]
)
print(summary.round(4).to_string(index=False))

# Curvas concatenadas (opcional): evolución TL + FT
def concat_histories(h1, h2):
    out = {}
    for k in h1.history:
        out[k] = list(h1.history[k]) + list(h2.history.get(k, []))
    return out

combo = concat_histories(history_tl, history_ft)
epochs = range(1, len(combo["loss"]) + 1)
split = len(history_tl.history["loss"]) + 0.5

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, combo["loss"], label="train")
plt.plot(epochs, combo["val_loss"], label="val")
plt.axvline(split, color="gray", linestyle="--", label="inicio FT")
plt.title("Loss concatenada (TL → FT)")
plt.xlabel("Época")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs, combo["accuracy"], label="train")
plt.plot(epochs, combo["val_accuracy"], label="val")
plt.axvline(split, color="gray", linestyle="--", label="inicio FT")
plt.title("Accuracy concatenada (TL → FT)")
plt.xlabel("Época")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


**Cómo leer el resultado**

- Si la fase 1 ya es fuerte y la fase 2 empeora: probablemente LR alta, demasiadas capas descongeladas o sobreajuste.
- Si la fase 2 mejora de forma estable: el fine-tuning parcial está aportando adaptación útil.
- CIFAR-10 es de baja resolución (32×32); aunque reescalemos, el techo de accuracy no será el de un dataset HD.


<a id="eleccion"></a>
<h2 style="color: #007ACC;">Cuándo usar cada enfoque</h2>


| Situación | Enfoque recomendado |
|-----------|---------------------|
| Pocos datos, dominio parecido a ImageNet | Transfer Learning (base congelada) |
| Datos moderados, dominio algo distinto | TL y luego fine-tuning parcial |
| Muchos datos, dominio muy distinto | Fine-tuning más agresivo (más capas / total) con LR baja |
| Recursos limitados | Congelar casi todo; cabeza pequeña; MobileNet / EfficientNet-lite |

Reglas prácticas:

1. **Siempre** entrena primero la cabeza con la base congelada.
2. Usa LR **más baja** al descongelar.
3. Descongela por **bloques**, no capas sueltas al azar.
4. Vigila `val_loss`: el fine-tuning puede sobreajustar rápido.


<a id="problemas"></a>
<h2 style="color: #007ACC;">Problemas comunes</h2>


- **Overfitting:** dropout, augmentation, early stopping, descongelar menos capas.
- **Underfitting:** LR demasiado baja, cabeza demasiado simple, o base demasiado congelada para un dominio lejano.
- **Colapso en accuracy ≈ 0.5:** suele indicar labels/loss incompatibles (p. ej. `softmax` de 2 clases con `binary_crossentropy`, o `argmax` sobre una sola neurona sigmoid), inicializaciones extremas o preprocesado incorrecto.
- **Olvido catastrófico del preentrenamiento:** LR alta al fine-tunear o descongelar todo demasiado pronto.
- **Preprocesado incorrecto:** cada familia (`VGG`, `MobileNet`, `ResNet`) tiene su `preprocess_input`; no basta con dividir entre 255 en todos los casos.
- **Desbalance de clases:** `class_weight`, re-muestreo o métricas más allá del accuracy (F1, recall).


<a id="reference"></a>
<h2 style="color: #007ACC;">Referencias</h2>


- Chollet, F. — *Transfer learning & fine-tuning* (Keras): https://keras.io/guides/transfer_learning/
- Tutorial TensorFlow — Transfer learning and fine-tuning: https://www.tensorflow.org/tutorials/images/transfer_learning
- MobileNetV2 (Sandler et al., 2018): https://arxiv.org/abs/1801.04381
- CIFAR-10: https://www.cs.toronto.edu/~kriz/cifar.html
